In [2]:
# Supplementary Audit — 00. Load source tables

from pathlib import Path
import pandas as pd
import numpy as np
import os
import re

# 1. Project directories
AUDIT_PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")

AUDIT_PROCESSED_DIR = (
    AUDIT_PROJECT_ROOT / "Data" / "processed"
)

NB03_OUTPUT_DIR = (
    AUDIT_PROCESSED_DIR / "nb03_lesion_dataset"
)

# 2. Load annotation mapping từ Notebook 03
annotation_path = (
    NB03_OUTPUT_DIR / "lesion_annotation_mapping.csv"
)

nb04_lesion_annotation_mapping_df = pd.read_csv(
    annotation_path,
    low_memory=False
)

# 3. Load ROI mapping từ Notebook 03
roi_mapping_path = (
    NB03_OUTPUT_DIR / "lesion_roi_mapping.csv"
)

nb04_lesion_roi_mapping_df = pd.read_csv(
    roi_mapping_path,
    low_memory=False
)

# Alias để tương thích với code A4
bbox_half_open_df = nb04_lesion_roi_mapping_df.copy()

# 4. Load DICOM role index từ Notebook 02
role_index_path = (
    AUDIT_PROCESSED_DIR / "nb02_instance_role_final.csv"
)

nb02_instance_role_df = pd.read_csv(
    role_index_path,
    dtype=str,
    low_memory=False
)

# 5. Kiểm tra đầu vào
print("=== SUPPLEMENTARY AUDIT INPUT CHECK ===")

for name, df in [
    ("Annotation mapping", nb04_lesion_annotation_mapping_df),
    ("ROI mapping", nb04_lesion_roi_mapping_df),
    ("DICOM role index", nb02_instance_role_df)
]:
    print(f"{name}: {df.shape}")

print("\n=== REQUIRED COLUMN CHECK ===")

required_columns = {
    "annotation_mapping": {
        "annotation_id",
        "patient_id",
        "full_image_csv_path",
        "crop_image_csv_path",
        "roi_mask_csv_path"
    },
    "roi_mapping": {
        "annotation_id",
        "roi_mask_file_path",
        "mask_quality_status"
    },
    "dicom_role_index": {
        "study_uid",
        "series_uid",
        "SOPInstanceUID",
        "file_path",
        "SeriesDescription",
        "proposed_role_final",
        "final_review_status"
    }
}

loaded_tables = {
    "annotation_mapping": nb04_lesion_annotation_mapping_df,
    "roi_mapping": nb04_lesion_roi_mapping_df,
    "dicom_role_index": nb02_instance_role_df
}

for name, required in required_columns.items():

    missing = required - set(loaded_tables[name].columns)

    print(
        f"{name}:",
        "OK" if not missing else f"MISSING {missing}"
    )

    if missing:
        raise ValueError(
            f"Bảng {name} thiếu cột: {missing}"
        )

print("\nKiểm tra hoàn tất.")

=== SUPPLEMENTARY AUDIT INPUT CHECK ===
Annotation mapping: (3568, 51)
ROI mapping: (3568, 87)
DICOM role index: (10239, 16)

=== REQUIRED COLUMN CHECK ===
annotation_mapping: OK
roi_mapping: OK
dicom_role_index: OK

Kiểm tra hoàn tất.


In [3]:
# Supplementary Audit A1 — CSV path vs DICOM role

import re
import os
from pathlib import Path, PurePosixPath
import pandas as pd
import numpy as np

# 1. Kiểm tra đầu vào
source_fields = {
    "full_image_csv_path": "full_image_candidate",
    "crop_image_csv_path": "crop_candidate",
    "roi_mask_csv_path": "roi_mask_candidate"
}

required_annotation_cols = {
    "annotation_id",
    *source_fields.keys()
}

required_dicom_cols = {
    "study_uid",
    "series_uid",
    "SOPInstanceUID",
    "file_path",
    "SeriesDescription",
    "proposed_role_final",
    "final_review_status"
}

assert required_annotation_cols.issubset(
    nb04_lesion_annotation_mapping_df.columns
)

assert required_dicom_cols.issubset(
    nb02_instance_role_df.columns
)

assert nb04_lesion_annotation_mapping_df[
    "annotation_id"
].is_unique


# 2. Chuẩn hóa DICOM index
role_index = nb02_instance_role_df.copy()

for col in [
    "study_uid",
    "series_uid",
    "SOPInstanceUID",
    "proposed_role_final"
]:
    role_index[col] = (
        role_index[col]
        .where(role_index[col].notna(), None)
    )

    role_index[col] = role_index[col].apply(
        lambda x: str(x).strip() if x is not None else None
    )


# 3. Đưa ba trường đường dẫn thành dạng long
meta_cols = ["annotation_id"]

if "patient_id" in nb04_lesion_annotation_mapping_df.columns:
    meta_cols.append("patient_id")

source_long_df = (
    nb04_lesion_annotation_mapping_df[
        meta_cols + list(source_fields.keys())
    ]
    .melt(
        id_vars=meta_cols,
        value_vars=list(source_fields.keys()),
        var_name="source_field",
        value_name="source_path"
    )
)

source_long_df["expected_role"] = (
    source_long_df["source_field"].map(source_fields)
)


# 4. Trích Study UID và Series UID
uid_pattern = re.compile(r"^\d+(?:\.\d+)+$")


def parse_source_path(value):

    if pd.isna(value):
        return {
            "uid_component_count": 0,
            "path_study_uid": None,
            "path_series_uid": None
        }

    parts = PurePosixPath(
        str(value).strip().replace("\\", "/")
    ).parts

    uids = [
        part.strip()
        for part in parts
        if uid_pattern.fullmatch(part.strip())
    ]

    return {
        "uid_component_count": len(uids),
        "path_study_uid": uids[0] if len(uids) >= 1 else None,
        "path_series_uid": uids[1] if len(uids) >= 2 else None
    }


parsed_paths = pd.DataFrame(
    source_long_df["source_path"].apply(
        parse_source_path
    ).tolist(),
    index=source_long_df.index
)

source_long_df = pd.concat(
    [source_long_df, parsed_paths],
    axis=1
)


# 5. Tóm tắt vai trò của từng Series
def unique_values(values):

    return tuple(sorted({
        str(value).strip()
        for value in values
        if pd.notna(value) and str(value).strip()
    }))


series_summary = (
    role_index
    .groupby(["study_uid", "series_uid"])
    .agg(
        series_instance_count=("SOPInstanceUID", "nunique"),
        actual_roles=("proposed_role_final", unique_values),
        series_descriptions=("SeriesDescription", unique_values)
    )
    .reset_index()
    .rename(columns={
        "study_uid": "path_study_uid",
        "series_uid": "path_series_uid"
    })
)


# 6. Thống kê instance theo từng role trong Series
series_role_detail = (
    role_index[
        role_index["proposed_role_final"].notna()
    ]
    .groupby(
        ["study_uid", "series_uid", "proposed_role_final"]
    )
    .agg(
        expected_role_instance_count=("SOPInstanceUID", "nunique"),
        expected_role_review_statuses=(
            "final_review_status", unique_values
        ),
        expected_role_descriptions=(
            "SeriesDescription", unique_values
        ),
        expected_role_file_paths=("file_path", unique_values)
    )
    .reset_index()
    .rename(columns={
        "study_uid": "path_study_uid",
        "series_uid": "path_series_uid",
        "proposed_role_final": "expected_role"
    })
)


# 7. Ghép kết quả audit
path_role_audit_df = source_long_df.merge(
    series_summary,
    on=["path_study_uid", "path_series_uid"],
    how="left",
    validate="many_to_one"
)

path_role_audit_df = path_role_audit_df.merge(
    series_role_detail,
    on=[
        "path_study_uid",
        "path_series_uid",
        "expected_role"
    ],
    how="left",
    validate="many_to_one"
)

path_role_audit_df["expected_role_instance_count"] = (
    path_role_audit_df["expected_role_instance_count"]
    .fillna(0)
    .astype(int)
)


# 8. Kiểm tra mô tả Series
expected_description = {
    "full_image_candidate": "full mammogram",
    "crop_candidate": "cropped images",
    "roi_mask_candidate": "roi mask"
}


def check_description_match(row):

    keyword = expected_description[row["expected_role"]]

    descriptions = row["series_descriptions"]

    if not isinstance(descriptions, tuple):
        return False

    return any(
        keyword in description.lower()
        for description in descriptions
    )


path_role_audit_df["description_role_match"] = (
    path_role_audit_df.apply(
        check_description_match,
        axis=1
    )
)


# 9. Phân loại trạng thái
path_role_audit_df["audit_status"] = "role_mismatch"

path_role_audit_df.loc[
    path_role_audit_df["source_path"].isna(),
    "audit_status"
] = "missing_source_path"

path_role_audit_df.loc[
    path_role_audit_df["source_path"].notna()
    & (path_role_audit_df["uid_component_count"] != 2),
    "audit_status"
] = "invalid_uid_path"

path_role_audit_df.loc[
    (path_role_audit_df["uid_component_count"] == 2)
    & path_role_audit_df["series_instance_count"].isna(),
    "audit_status"
] = "series_not_found"

path_role_audit_df.loc[
    path_role_audit_df["expected_role_instance_count"] == 1,
    "audit_status"
] = "role_match"

path_role_audit_df.loc[
    path_role_audit_df["expected_role_instance_count"] > 1,
    "audit_status"
] = "role_match_multiple_instances"


print("Annotation:", nb04_lesion_annotation_mapping_df.shape[0])
print("Số dòng audit:", len(path_role_audit_df))

display(
    path_role_audit_df[
        [
            "annotation_id",
            "source_field",
            "expected_role",
            "path_study_uid",
            "path_series_uid",
            "actual_roles",
            "series_descriptions",
            "expected_role_instance_count",
            "expected_role_review_statuses",
            "description_role_match",
            "audit_status"
        ]
    ].head(12)
)

Annotation: 3568
Số dòng audit: 10704


,annotation_id,source_field,expected_role,path_study_uid,path_series_uid,actual_roles,series_descriptions,expected_role_instance_count,expected_role_review_statuses,description_role_match,audit_status
0,calc_train__00001,full_image_csv_path,full_image_candidate,1.3.6.1.4.1.9590.100.1.2.408909860712120272633...,1.3.6.1.4.1.9590.100.1.2.474143160103683865197...,"(full_image_candidate,)","(full mammogram images,)",1,"(path_context_supported,)",True,role_match
1,calc_train__00002,full_image_csv_path,full_image_candidate,1.3.6.1.4.1.9590.100.1.2.427517897311902339923...,1.3.6.1.4.1.9590.100.1.2.250596608311207922527...,"(full_image_candidate,)","(full mammogram images,)",1,"(path_context_supported,)",True,role_match
2,calc_train__00003,full_image_csv_path,full_image_candidate,1.3.6.1.4.1.9590.100.1.2.201322325113694962619...,1.3.6.1.4.1.9590.100.1.2.228699627313487111012...,"(full_image_candidate,)","(full mammogram images,)",1,"(path_context_supported,)",True,role_match
3,calc_train__00004,full_image_csv_path,full_image_candidate,1.3.6.1.4.1.9590.100.1.2.370479499712916693322...,1.3.6.1.4.1.9590.100.1.2.104743410411133110629...,"(full_image_candidate,)","(full mammogram images,)",1,"(path_context_supported,)",True,role_match
4,calc_train__00005,full_image_csv_path,full_image_candidate,1.3.6.1.4.1.9590.100.1.2.162256682111885666305...,1.3.6.1.4.1.9590.100.1.2.406725628213826290127...,"(full_image_candidate,)","(full mammogram images,)",1,"(path_context_supported,)",True,role_match
5,calc_train__00006,full_image_csv_path,full_image_candidate,1.3.6.1.4.1.9590.100.1.2.162256682111885666305...,1.3.6.1.4.1.9590.100.1.2.406725628213826290127...,"(full_image_candidate,)","(full mammogram images,)",1,"(path_context_supported,)",True,role_match
6,calc_train__00007,full_image_csv_path,full_image_candidate,1.3.6.1.4.1.9590.100.1.2.162256682111885666305...,1.3.6.1.4.1.9590.100.1.2.406725628213826290127...,"(full_image_candidate,)","(full mammogram images,)",1,"(path_context_supported,)",True,role_match
7,calc_train__00008,full_image_csv_path,full_image_candidate,1.3.6.1.4.1.9590.100.1.2.841558427127656604294...,1.3.6.1.4.1.9590.100.1.2.147846893112398086048...,"(full_image_candidate,)","(full mammogram images,)",1,"(path_context_supported,)",True,role_match
8,calc_train__00009,full_image_csv_path,full_image_candidate,1.3.6.1.4.1.9590.100.1.2.841558427127656604294...,1.3.6.1.4.1.9590.100.1.2.147846893112398086048...,"(full_image_candidate,)","(full mammogram images,)",1,"(path_context_supported,)",True,role_match
9,calc_train__00010,full_image_csv_path,full_image_candidate,1.3.6.1.4.1.9590.100.1.2.841558427127656604294...,1.3.6.1.4.1.9590.100.1.2.147846893112398086048...,"(full_image_candidate,)","(full mammogram images,)",1,"(path_context_supported,)",True,role_match


In [4]:
# Supplementary Audit A2 — Path role summary

path_role_summary_df = (
    path_role_audit_df
    .groupby(
        ["source_field", "expected_role", "audit_status"],
        dropna=False
    )
    .agg(
        annotation_count=("annotation_id", "nunique"),
        patient_count=(
            "patient_id", "nunique"
        ) if "patient_id" in path_role_audit_df.columns
        else ("annotation_id", "nunique")
    )
    .reset_index()
)

print("=== SOURCE PATH ROLE SUMMARY ===")
display(path_role_summary_df)


# Những trường hợp không khớp hoàn toàn
path_role_exception_df = path_role_audit_df[
    ~path_role_audit_df["audit_status"].isin([
        "role_match",
        "role_match_multiple_instances"
    ])
].copy()

print("\n=== PATH ROLE EXCEPTIONS ===")
print("Số bản ghi cần rà soát:", len(path_role_exception_df))

display(
    path_role_exception_df[
        [
            "annotation_id",
            "source_field",
            "source_path",
            "expected_role",
            "actual_roles",
            "series_descriptions",
            "expected_role_instance_count",
            "expected_role_review_statuses",
            "description_role_match",
            "audit_status"
        ]
    ].head(50)
)


# Kiểm tra mức độ hỗ trợ của role review
print("\n=== ROLE REVIEW EVIDENCE ===")

display(
    path_role_audit_df[
        path_role_audit_df["expected_role_instance_count"] > 0
    ]
    .groupby(
        ["source_field", "expected_role_review_statuses"],
        dropna=False
    )
    .agg(
        annotation_count=("annotation_id", "nunique")
    )
    .reset_index()
)

=== SOURCE PATH ROLE SUMMARY ===


,source_field,expected_role,audit_status,annotation_count,patient_count
0,crop_image_csv_path,crop_candidate,role_match,3567,1566
1,crop_image_csv_path,crop_candidate,role_mismatch,1,1
2,full_image_csv_path,full_image_candidate,role_match,3568,1566
3,roi_mask_csv_path,roi_mask_candidate,role_match,3567,1566
4,roi_mask_csv_path,roi_mask_candidate,role_mismatch,1,1



=== PATH ROLE EXCEPTIONS ===
Số bản ghi cần rà soát: 2


,annotation_id,source_field,source_path,expected_role,actual_roles,series_descriptions,expected_role_instance_count,expected_role_review_statuses,description_role_match,audit_status
3912,calc_train__00345,crop_image_csv_path,Calc-Training_P_00474_LEFT_MLO_1/1.3.6.1.4.1.9...,crop_candidate,"(roi_mask_candidate,)","(ROI mask images,)",0,NaN,False,role_mismatch
7480,calc_train__00345,roi_mask_csv_path,Calc-Training_P_00474_LEFT_MLO_1/1.3.6.1.4.1.9...,roi_mask_candidate,"(crop_candidate,)","(cropped images,)",0,NaN,False,role_mismatch



=== ROLE REVIEW EVIDENCE ===


,source_field,expected_role_review_statuses,annotation_count
0,crop_image_csv_path,"(description_supported,)",3552
1,crop_image_csv_path,"(pair_pattern_supported,)",15
2,full_image_csv_path,"(path_context_supported,)",3568
3,roi_mask_csv_path,"(description_supported,)",3247
4,roi_mask_csv_path,"(inferred_needs_validation,)",305
5,roi_mask_csv_path,"(pair_pattern_supported,)",15


In [ ]:
# Supplementary Audit A3 — Crop / ROI swap detection

role_pivot_df = (
    path_role_audit_df
    .pivot(
        index="annotation_id",
        columns="source_field",
        values="actual_roles"
    )
)

status_pivot_df = (
    path_role_audit_df
    .pivot(
        index="annotation_id",
        columns="source_field",
        values="audit_status"
    )
)

def contains_role(value, target_role):

    return (
        isinstance(value, tuple)
        and target_role in value
    )


def has_only_role(value, target_role):

    return (
        isinstance(value, tuple)
        and len(value) == 1
        and value[0] == target_role
    )

swap_check_df = pd.DataFrame(index=role_pivot_df.index)

swap_check_df["full_actual_roles"] = (
    role_pivot_df["full_image_csv_path"]
)

swap_check_df["crop_actual_roles"] = (
    role_pivot_df["crop_image_csv_path"]
)

swap_check_df["roi_actual_roles"] = (
    role_pivot_df["roi_mask_csv_path"]
)

swap_check_df["full_path_role_mismatch"] = (
    ~swap_check_df["full_actual_roles"].apply(
        lambda x: contains_role(x, "full_image_candidate")
    )
)

swap_check_df["crop_path_points_to_roi_only"] = (
    swap_check_df["crop_actual_roles"].apply(
        lambda x: has_only_role(x, "roi_mask_candidate")
    )
)

swap_check_df["roi_path_points_to_crop_only"] = (
    swap_check_df["roi_actual_roles"].apply(
        lambda x: has_only_role(x, "crop_candidate")
    )
)

swap_check_df["possible_crop_roi_swap"] = (
    swap_check_df["crop_path_points_to_roi_only"]
    & swap_check_df["roi_path_points_to_crop_only"]
)

swap_check_df["crop_path_has_role_mismatch"] = (
    ~swap_check_df["crop_actual_roles"].apply(
        lambda x: contains_role(x, "crop_candidate")
    )
)

swap_check_df["roi_path_has_role_mismatch"] = (
    ~swap_check_df["roi_actual_roles"].apply(
        lambda x: contains_role(x, "roi_mask_candidate")
    )
)

swap_check_df = swap_check_df.reset_index()


# Thêm patient ID nếu có
if "patient_id" in nb04_lesion_annotation_mapping_df.columns:

    swap_check_df = swap_check_df.merge(
        nb04_lesion_annotation_mapping_df[
            ["annotation_id", "patient_id"]
        ],
        on="annotation_id",
        how="left",
        validate="one_to_one"
    )

print("=== CROP / ROI SWAP AUDIT SUMMARY ===")

display(
    pd.DataFrame({
        "metric": [
            "Total annotations",
            "Full Image path role mismatch",
            "Crop path role mismatch",
            "ROI path role mismatch",
            "Crop path points to ROI only",
            "ROI path points to Crop only",
            "Possible Crop / ROI swap"
        ],
        "count": [
            len(swap_check_df),
            int(swap_check_df["full_path_role_mismatch"].sum()),
            int(swap_check_df["crop_path_has_role_mismatch"].sum()),
            int(swap_check_df["roi_path_has_role_mismatch"].sum()),
            int(swap_check_df["crop_path_points_to_roi_only"].sum()),
            int(swap_check_df["roi_path_points_to_crop_only"].sum()),
            int(swap_check_df["possible_crop_roi_swap"].sum())
        ]
    })
)

print("\n=== POSSIBLE SWAP CASES ===")

display(
    swap_check_df[
        swap_check_df["possible_crop_roi_swap"]
        | swap_check_df["full_path_role_mismatch"]
        | swap_check_df["crop_path_has_role_mismatch"]
        | swap_check_df["roi_path_has_role_mismatch"]
    ].head(50)
)

=== CROP / ROI SWAP AUDIT SUMMARY ===


,metric,count
0,Total annotations,3568
1,Full Image path role mismatch,0
2,Crop path role mismatch,1
3,ROI path role mismatch,1
4,Crop path points to ROI only,1
5,ROI path points to Crop only,1
6,Possible Crop / ROI swap,1



=== POSSIBLE SWAP CASES ===


,annotation_id,full_actual_roles,crop_actual_roles,roi_actual_roles,full_path_role_mismatch,crop_path_points_to_roi_only,roi_path_points_to_crop_only,possible_crop_roi_swap,crop_path_has_role_mismatch,roi_path_has_role_mismatch,patient_id
670,calc_train__00345,"(full_image_candidate,)","(roi_mask_candidate,)","(crop_candidate,)",False,True,True,True,True,True,P_00474


In [ ]:
# Supplementary Audit A4 — Physical ROI mapping audit

# 1. Chuẩn hóa đường dẫn Windows
def normalize_absolute_path(value):

    if pd.isna(value):
        return None

    return os.path.normcase(
        os.path.normpath(str(value).strip())
    )


role_physical_lookup_df = role_index[
    [
        "study_uid",
        "series_uid",
        "SOPInstanceUID",
        "file_path",
        "SeriesDescription",
        "proposed_role_final",
        "final_review_status"
    ]
].copy()

role_physical_lookup_df["_normalized_path"] = (
    role_physical_lookup_df["file_path"]
    .apply(normalize_absolute_path)
)


# 2. Kiểm tra tính duy nhất của đường dẫn DICOM
duplicate_physical_paths = (
    role_physical_lookup_df[
        role_physical_lookup_df["_normalized_path"].duplicated(
            keep=False
        )
    ]
)

print("DICOM path trùng:", len(duplicate_physical_paths))

if not duplicate_physical_paths.empty:
    display(duplicate_physical_paths.head(20))
    raise ValueError(
        "Đường dẫn DICOM bị trùng, chưa thể audit ROI vật lý."
    )

# 3. Lập bảng ROI vật lý từ Notebook 03
roi_physical_audit_df = bbox_half_open_df[
    [
        "annotation_id",
        "roi_mask_file_path",
        "mask_quality_status"
    ]
].copy()

roi_physical_audit_df["_normalized_path"] = (
    roi_physical_audit_df["roi_mask_file_path"]
    .apply(normalize_absolute_path)
)

roi_physical_audit_df["_file_exists"] = (
    roi_physical_audit_df["roi_mask_file_path"].apply(
        lambda x: Path(str(x)).is_file()
        if pd.notna(x) else False
    )
)

# 4. Đối chiếu đường dẫn vật lý với DICOM index
roi_physical_audit_df = roi_physical_audit_df.merge(
    role_physical_lookup_df.drop(
        columns=["file_path"]
    ),
    on="_normalized_path",
    how="left",
    validate="many_to_one"
)

# 5. Phân loại trạng thái ROI vật lý
roi_physical_audit_df["physical_roi_status"] = (
    "roi_role_match"
)

roi_physical_audit_df.loc[
    ~roi_physical_audit_df["_file_exists"],
    "physical_roi_status"
] = "file_not_found"

roi_physical_audit_df.loc[
    roi_physical_audit_df["_file_exists"]
    & roi_physical_audit_df["proposed_role_final"].isna(),
    "physical_roi_status"
] = "not_in_dicom_index"

roi_physical_audit_df.loc[
    roi_physical_audit_df["proposed_role_final"].notna()
    & (
        roi_physical_audit_df["proposed_role_final"]
        != "roi_mask_candidate"
    ),
    "physical_roi_status"
] = "physical_role_mismatch"

# 6. Lấy UID của Crop CSV và ROI CSV để so sánh
crop_csv_uid_df = path_role_audit_df[
    path_role_audit_df["source_field"] == "crop_image_csv_path"
][
    [
        "annotation_id",
        "path_study_uid",
        "path_series_uid",
        "actual_roles",
        "audit_status"
    ]
].rename(columns={
    "path_study_uid": "crop_csv_study_uid",
    "path_series_uid": "crop_csv_series_uid",
    "actual_roles": "crop_csv_actual_roles",
    "audit_status": "crop_csv_audit_status"
})

roi_csv_uid_df = path_role_audit_df[
    path_role_audit_df["source_field"] == "roi_mask_csv_path"
][
    [
        "annotation_id",
        "path_study_uid",
        "path_series_uid",
        "actual_roles",
        "audit_status"
    ]
].rename(columns={
    "path_study_uid": "roi_csv_study_uid",
    "path_series_uid": "roi_csv_series_uid",
    "actual_roles": "roi_csv_actual_roles",
    "audit_status": "roi_csv_audit_status"
})

roi_physical_audit_df = roi_physical_audit_df.merge(
    crop_csv_uid_df,
    on="annotation_id",
    how="left",
    validate="one_to_one"
).merge(
    roi_csv_uid_df,
    on="annotation_id",
    how="left",
    validate="one_to_one"
)

# 7. So sánh Series vật lý với hai đường dẫn CSV
roi_physical_audit_df["physical_roi_matches_crop_csv_series"] = (
    (roi_physical_audit_df["study_uid"]
     == roi_physical_audit_df["crop_csv_study_uid"])
    &
    (roi_physical_audit_df["series_uid"]
     == roi_physical_audit_df["crop_csv_series_uid"])
)

roi_physical_audit_df["physical_roi_matches_roi_csv_series"] = (
    (roi_physical_audit_df["study_uid"]
     == roi_physical_audit_df["roi_csv_study_uid"])
    &
    (roi_physical_audit_df["series_uid"]
     == roi_physical_audit_df["roi_csv_series_uid"])
)

# 8. Báo cáo
print("=== PHYSICAL ROI ROLE SUMMARY ===")

display(
    roi_physical_audit_df.groupby(
        ["physical_roi_status", "proposed_role_final"],
        dropna=False
    ).size().reset_index(name="annotation_count")
)

print("\n=== PHYSICAL ROI vs CSV PATH SUMMARY ===")

display(
    pd.DataFrame({
        "metric": [
            "Total ROI mappings",
            "Physical ROI file exists",
            "Physical ROI is in DICOM index",
            "Physical ROI classified as ROI mask",
            "Physical ROI matches crop CSV Series",
            "Physical ROI matches ROI CSV Series"
        ],
        "count": [
            len(roi_physical_audit_df),
            int(roi_physical_audit_df["_file_exists"].sum()),
            int(roi_physical_audit_df["study_uid"].notna().sum()),
            int(
                (
                    roi_physical_audit_df["proposed_role_final"]
                    == "roi_mask_candidate"
                ).sum()
            ),
            int(
                roi_physical_audit_df[
                    "physical_roi_matches_crop_csv_series"
                ].sum()
            ),
            int(
                roi_physical_audit_df[
                    "physical_roi_matches_roi_csv_series"
                ].sum()
            )
        ]
    })
)

print("\n=== PHYSICAL ROI EXCEPTIONS ===")

display(
    roi_physical_audit_df[
        (roi_physical_audit_df["physical_roi_status"] != "roi_role_match")
        | (
            ~roi_physical_audit_df[
                "physical_roi_matches_roi_csv_series"
            ].fillna(False)
        )
    ][
        [
            "annotation_id",
            "roi_mask_file_path",
            "SeriesDescription",
            "proposed_role_final",
            "crop_csv_actual_roles",
            "roi_csv_actual_roles",
            "physical_roi_matches_crop_csv_series",
            "physical_roi_matches_roi_csv_series",
            "physical_roi_status"
        ]
    ].head(50)
)

DICOM path trùng: 0
=== PHYSICAL ROI ROLE SUMMARY ===


,physical_roi_status,proposed_role_final,annotation_count
0,roi_role_match,roi_mask_candidate,3568



=== PHYSICAL ROI vs CSV PATH SUMMARY ===


,metric,count
0,Total ROI mappings,3568
1,Physical ROI file exists,3568
2,Physical ROI is in DICOM index,3568
3,Physical ROI classified as ROI mask,3568
4,Physical ROI matches crop CSV Series,3465
5,Physical ROI matches ROI CSV Series,3567



=== PHYSICAL ROI EXCEPTIONS ===


,annotation_id,roi_mask_file_path,SeriesDescription,proposed_role_final,crop_csv_actual_roles,roi_csv_actual_roles,physical_roi_matches_crop_csv_series,physical_roi_matches_roi_csv_series,physical_roi_status
344,calc_train__00345,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,ROI mask images,roi_mask_candidate,"(roi_mask_candidate,)","(crop_candidate,)",True,False,roi_role_match


In [7]:
# Supplementary Audit A6 — Cross-check physical ROI and CSV Series

crop_uid_df = path_role_audit_df[
    path_role_audit_df["source_field"] == "crop_image_csv_path"
][
    ["annotation_id", "path_study_uid", "path_series_uid", "actual_roles"]
].rename(columns={
    "path_study_uid": "crop_study_uid",
    "path_series_uid": "crop_series_uid",
    "actual_roles": "crop_actual_roles"
})

roi_uid_df = path_role_audit_df[
    path_role_audit_df["source_field"] == "roi_mask_csv_path"
][
    ["annotation_id", "path_study_uid", "path_series_uid", "actual_roles"]
].rename(columns={
    "path_study_uid": "csv_roi_study_uid",
    "path_series_uid": "csv_roi_series_uid",
    "actual_roles": "csv_roi_actual_roles"
})

physical_crosscheck_df = (
    roi_physical_audit_df[
        [
            "annotation_id",
            "study_uid",
            "series_uid",
            "proposed_role_final",
            "final_review_status",
            "_file_exists"
        ]
    ]
    .merge(
        crop_uid_df,
        on="annotation_id",
        validate="one_to_one"
    )
    .merge(
        roi_uid_df,
        on="annotation_id",
        validate="one_to_one"
    )
)

physical_crosscheck_df["roi_physical_matches_crop_series"] = (
    (physical_crosscheck_df["study_uid"] ==
     physical_crosscheck_df["crop_study_uid"])
    &
    (physical_crosscheck_df["series_uid"] ==
     physical_crosscheck_df["crop_series_uid"])
)

physical_crosscheck_df["roi_physical_matches_roi_series"] = (
    (physical_crosscheck_df["study_uid"] ==
     physical_crosscheck_df["csv_roi_study_uid"])
    &
    (physical_crosscheck_df["series_uid"] ==
     physical_crosscheck_df["csv_roi_series_uid"])
)

physical_crosscheck_df["crop_and_roi_csv_same_series"] = (
    (physical_crosscheck_df["crop_study_uid"] ==
     physical_crosscheck_df["csv_roi_study_uid"])
    &
    (physical_crosscheck_df["crop_series_uid"] ==
     physical_crosscheck_df["csv_roi_series_uid"])
)

# Phân loại quan hệ
def classify_series_relation(row):

    crop_match = row["roi_physical_matches_crop_series"]
    roi_match = row["roi_physical_matches_roi_series"]

    if crop_match and roi_match:
        return "physical_roi_matches_both"

    if crop_match:
        return "physical_roi_matches_crop_only"

    if roi_match:
        return "physical_roi_matches_roi_only"

    return "physical_roi_matches_neither"


physical_crosscheck_df["series_relation"] = (
    physical_crosscheck_df.apply(
        classify_series_relation,
        axis=1
    )
)

print("=== PHYSICAL ROI / CSV SERIES RELATION ===")

display(
    physical_crosscheck_df.groupby(
        "series_relation"
    ).size().reset_index(name="annotation_count")
)

print("\n=== CROP / ROI CSV SAME SERIES ===")

display(
    physical_crosscheck_df[
        "crop_and_roi_csv_same_series"
    ].value_counts().rename_axis(
        "same_series"
    ).reset_index(name="annotation_count")
)

print("\n=== PHYSICAL ROI REVIEW EVIDENCE ===")

display(
    physical_crosscheck_df.groupby(
        ["proposed_role_final", "final_review_status"],
        dropna=False
    ).size().reset_index(name="annotation_count")
)

print("\n=== NON-STANDARD SERIES RELATIONS ===")

display(
    physical_crosscheck_df[
        (physical_crosscheck_df["series_relation"]
         != "physical_roi_matches_roi_only")
        | (~physical_crosscheck_df["_file_exists"])
    ][
        [
            "annotation_id",
            "crop_actual_roles",
            "csv_roi_actual_roles",
            "crop_and_roi_csv_same_series",
            "series_relation",
            "proposed_role_final",
            "final_review_status",
            "_file_exists"
        ]
    ].head(30)
)

=== PHYSICAL ROI / CSV SERIES RELATION ===


,series_relation,annotation_count
0,physical_roi_matches_both,3464
1,physical_roi_matches_crop_only,1
2,physical_roi_matches_roi_only,103



=== CROP / ROI CSV SAME SERIES ===


,same_series,annotation_count
0,True,3464
1,False,104



=== PHYSICAL ROI REVIEW EVIDENCE ===


,proposed_role_final,final_review_status,annotation_count
0,roi_mask_candidate,description_supported,3248
1,roi_mask_candidate,inferred_needs_validation,305
2,roi_mask_candidate,pair_pattern_supported,15



=== NON-STANDARD SERIES RELATIONS ===


,annotation_id,crop_actual_roles,csv_roi_actual_roles,crop_and_roi_csv_same_series,series_relation,proposed_role_final,final_review_status,_file_exists
0,calc_train__00001,"(crop_candidate, roi_mask_candidate)","(crop_candidate, roi_mask_candidate)",True,physical_roi_matches_both,roi_mask_candidate,description_supported,True
1,calc_train__00002,"(crop_candidate, roi_mask_candidate)","(crop_candidate, roi_mask_candidate)",True,physical_roi_matches_both,roi_mask_candidate,description_supported,True
2,calc_train__00003,"(crop_candidate, roi_mask_candidate)","(crop_candidate, roi_mask_candidate)",True,physical_roi_matches_both,roi_mask_candidate,description_supported,True
3,calc_train__00004,"(crop_candidate, roi_mask_candidate)","(crop_candidate, roi_mask_candidate)",True,physical_roi_matches_both,roi_mask_candidate,description_supported,True
4,calc_train__00005,"(crop_candidate, roi_mask_candidate)","(crop_candidate, roi_mask_candidate)",True,physical_roi_matches_both,roi_mask_candidate,description_supported,True
5,calc_train__00006,"(crop_candidate, roi_mask_candidate)","(crop_candidate, roi_mask_candidate)",True,physical_roi_matches_both,roi_mask_candidate,description_supported,True
6,calc_train__00007,"(crop_candidate, roi_mask_candidate)","(crop_candidate, roi_mask_candidate)",True,physical_roi_matches_both,roi_mask_candidate,description_supported,True
7,calc_train__00008,"(crop_candidate, roi_mask_candidate)","(crop_candidate, roi_mask_candidate)",True,physical_roi_matches_both,roi_mask_candidate,description_supported,True
8,calc_train__00009,"(crop_candidate, roi_mask_candidate)","(crop_candidate, roi_mask_candidate)",True,physical_roi_matches_both,roi_mask_candidate,description_supported,True
9,calc_train__00010,"(crop_candidate, roi_mask_candidate)","(crop_candidate, roi_mask_candidate)",True,physical_roi_matches_both,roi_mask_candidate,description_supported,True


In [8]:
# Supplementary Audit A7 — Build verified visualization mapping

# 1. Lấy danh sách annotation có dấu hiệu đảo Crop / ROI
swap_ids = set(
    swap_check_df.loc[
        swap_check_df["possible_crop_roi_swap"],
        "annotation_id"
    ]
)

print("Số annotation cần điều chỉnh mapping:", len(swap_ids))


# 2. Chuẩn bị DICOM role lookup
mapping_role_lookup_df = role_index[
    [
        "study_uid",
        "series_uid",
        "SOPInstanceUID",
        "file_path",
        "SeriesDescription",
        "proposed_role_final",
        "final_review_status"
    ]
].copy()

mapping_role_lookup_df = mapping_role_lookup_df[
    mapping_role_lookup_df["proposed_role_final"].isin([
        "full_image_candidate",
        "crop_candidate"
    ])
]

role_key_cols = [
    "study_uid",
    "series_uid",
    "proposed_role_final"
]

if mapping_role_lookup_df.duplicated(role_key_cols).any():
    raise ValueError(
        "Có Study + Series + Role không duy nhất. "
        "Không tự chọn DICOM instance."
    )


# 3. Chuẩn bị thông tin annotation
annotation_meta = (
    nb04_lesion_annotation_mapping_df
    .set_index("annotation_id")
)

visual_records = []

# 4. Tạo các quan hệ Full Image và Cropped Image
for _, row in nb04_lesion_annotation_mapping_df.iterrows():

    annotation_id = row["annotation_id"]

    image_definitions = [
        ("full_image", "full_image_csv_path", "full_image_candidate"),
        ("cropped_image", "crop_image_csv_path", "crop_candidate")
    ]

    for image_role, original_field, expected_role in image_definitions:

        applied_field = original_field

        # Chỉ xử lý annotation đã được audit xác nhận
        if (
            annotation_id in swap_ids
            and image_role == "cropped_image"
        ):
            applied_field = "roi_mask_csv_path"

        source_path = row[applied_field]

        parsed = parse_source_path(source_path)

        visual_records.append({
            "annotation_id": annotation_id,
            "image_role": image_role,
            "original_source_field": original_field,
            "applied_source_field": applied_field,
            "original_csv_path": row[original_field],
            "mapping_csv_path": source_path,
            "path_study_uid": parsed["path_study_uid"],
            "path_series_uid": parsed["path_series_uid"],
            "expected_role": expected_role,
            "mapping_adjusted": applied_field != original_field
        })


visual_source_df = pd.DataFrame(visual_records)


# 5. Ghép đúng DICOM instance theo Study + Series + Role
visual_image_mapping_df = visual_source_df.merge(
    mapping_role_lookup_df,
    left_on=[
        "path_study_uid",
        "path_series_uid",
        "expected_role"
    ],
    right_on=[
        "study_uid",
        "series_uid",
        "proposed_role_final"
    ],
    how="left",
    validate="many_to_one"
)

visual_image_mapping_df["_image_path"] = (
    visual_image_mapping_df["file_path"].apply(
        lambda x: Path(str(x)) if pd.notna(x) else None
    )
)

visual_image_mapping_df["_path_exists"] = (
    visual_image_mapping_df["_image_path"].apply(
        lambda x: x.is_file() if x is not None else False
    )
)

visual_image_mapping_df["_mapping_status"] = "resolved"

visual_image_mapping_df.loc[
    visual_image_mapping_df["SOPInstanceUID"].isna(),
    "_mapping_status"
] = "role_not_resolved"

visual_image_mapping_df.loc[
    ~visual_image_mapping_df["_path_exists"],
    "_mapping_status"
] = "file_not_found"


# 6. Tạo bảng ROI vật lý từ kết quả A4
visual_roi_mapping_df = roi_physical_audit_df[
    [
        "annotation_id",
        "roi_mask_file_path",
        "study_uid",
        "series_uid",
        "SOPInstanceUID",
        "SeriesDescription",
        "proposed_role_final",
        "final_review_status",
        "_file_exists",
        "physical_roi_status"
    ]
].copy()

visual_roi_mapping_df = visual_roi_mapping_df.rename(
    columns={
        "roi_mask_file_path": "_image_path",
        "_file_exists": "_path_exists",
        "proposed_role_final": "expected_role"
    }
)

visual_roi_mapping_df["image_role"] = "roi_mask"


# 7. Kiểm tra số lượng và tính đầy đủ
print("\n=== VERIFIED IMAGE MAPPING ===")

display(
    visual_image_mapping_df.groupby(
        ["image_role", "_mapping_status"]
    ).agg(
        relation_count=("annotation_id", "size"),
        unique_annotations=("annotation_id", "nunique"),
        existing_files=("_path_exists", "sum")
    ).reset_index()
)

print("\n=== VERIFIED ROI MAPPING ===")

display(
    visual_roi_mapping_df.groupby(
        ["physical_roi_status", "final_review_status"],
        dropna=False
    ).size().reset_index(name="annotation_count")
)


# 8. Kiểm tra override và tính toàn vẹn
print("\n=== MAPPING ADJUSTMENT ===")

display(
    visual_image_mapping_df[
        visual_image_mapping_df["mapping_adjusted"]
    ][
        [
            "annotation_id",
            "image_role",
            "original_source_field",
            "applied_source_field",
            "expected_role",
            "SeriesDescription",
            "file_path"
        ]
    ]
)

expected_annotations = (
    nb04_lesion_annotation_mapping_df["annotation_id"].nunique()
)

image_mapping_ready = (
    len(visual_image_mapping_df) == expected_annotations * 2
    and visual_image_mapping_df["_path_exists"].all()
    and visual_image_mapping_df["SOPInstanceUID"].notna().all()
    and visual_image_mapping_df.groupby("image_role")[
        "annotation_id"
    ].nunique().eq(expected_annotations).all()
)

roi_mapping_ready = (
    len(visual_roi_mapping_df) == expected_annotations
    and visual_roi_mapping_df["_path_exists"].all()
    and (
        visual_roi_mapping_df["expected_role"]
        == "roi_mask_candidate"
    ).all()
)

print("\n=== FINAL AUDIT STATUS ===")
print("Image mapping ready:", image_mapping_ready)
print("ROI mapping ready:", roi_mapping_ready)
print("Total Full/Cropped relations:", len(visual_image_mapping_df))
print("Total ROI mappings:", len(visual_roi_mapping_df))

if not image_mapping_ready or not roi_mapping_ready:
    raise RuntimeError(
        "Mapping chưa đạt điều kiện. Chưa chuyển sang trực quan hóa."
    )

Số annotation cần điều chỉnh mapping: 1

=== VERIFIED IMAGE MAPPING ===


,image_role,_mapping_status,relation_count,unique_annotations,existing_files
0,cropped_image,resolved,3568,3568,3568
1,full_image,resolved,3568,3568,3568



=== VERIFIED ROI MAPPING ===


,physical_roi_status,final_review_status,annotation_count
0,roi_role_match,description_supported,3248
1,roi_role_match,inferred_needs_validation,305
2,roi_role_match,pair_pattern_supported,15



=== MAPPING ADJUSTMENT ===


,annotation_id,image_role,original_source_field,applied_source_field,expected_role,SeriesDescription,file_path
689,calc_train__00345,cropped_image,crop_image_csv_path,roi_mask_csv_path,crop_candidate,cropped images,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...



=== FINAL AUDIT STATUS ===
Image mapping ready: True
ROI mapping ready: True
Total Full/Cropped relations: 7136
Total ROI mappings: 3568


In [9]:
# Supplementary Audit — Save verified mapping

from pathlib import Path
from datetime import datetime

audit_save_root = Path(
    r"C:\Breast-Cancer-CBIS-DDSM\Data\processed"
)

# Tạo thư mục riêng, không ghi đè kết quả cũ
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

audit_output_dir = (
    audit_save_root /
    f"supplementary_image_role_audit_{timestamp}"
)

audit_output_dir.mkdir(
    parents=True,
    exist_ok=False
)

# Các bảng cần cho Notebook 04
files_to_save = {
    "visual_image_mapping.csv": visual_image_mapping_df,
    "visual_roi_mapping.csv": visual_roi_mapping_df,
    "path_role_audit.csv": path_role_audit_df,
    "path_role_exceptions.csv": path_role_exception_df,
    "swap_check.csv": swap_check_df,
    "roi_physical_audit.csv": roi_physical_audit_df,
    "physical_crosscheck.csv": physical_crosscheck_df
}

for filename, df in files_to_save.items():

    df.to_csv(
        audit_output_dir / filename,
        index=False,
        encoding="utf-8-sig"
    )

print("=== AUDIT FILES SAVED ===")
print("Output directory:", audit_output_dir)

for filename in files_to_save:
    file_path = audit_output_dir / filename
    print(filename, "|", file_path.stat().st_size, "bytes")

print("\n[OK] Đã lưu các bảng kiểm định.")

=== AUDIT FILES SAVED ===
Output directory: C:\Breast-Cancer-CBIS-DDSM\Data\processed\supplementary_image_role_audit_20261105_002929_370953
visual_image_mapping.csv | 8309130 bytes
visual_roi_mapping.csv | 1646716 bytes
path_role_audit.csv | 7413493 bytes
path_role_exceptions.csv | 1207 bytes
swap_check.csv | 627117 bytes
roi_physical_audit.csv | 3606733 bytes
physical_crosscheck.csv | 2068580 bytes

[OK] Đã lưu các bảng kiểm định.
